In [ ]:
import numpy as np
from cellpose import models, core, io
from spotiflow.model import Spotiflow
from pathlib import Path
import os
import apoc
import pandas as pd
import pyclesperanto_prototype as cle
from utils import (
    list_images,
    read_image,
    brightfield_correction,
    detect_infected_cells,
    extract_mtb_regionprops,
    map_bacterial_location,
    extract_cell_features,
    detect_infection_load,
    puncta_detection,
)

io.logger_setup()  # run this to get printing of progress

# Check if notebook has GPU access
if core.use_gpu() == False:
    raise ImportError("No GPU access, change your runtime")

# Activate .cle GPU acceleration
cle.select_device("RTX")

# Load pre-trained Cellpose-SAM and Spotiflow models
model = models.CellposeModel(gpu=True)
spotiflow_model = Spotiflow.from_pretrained("general")

In [ ]:
# Load pretrained Object Classifier for Mycobacterium infection detection
mtb_cl_filename = "./pretrained_classifiers/no_nuclei_signal/siMtb screen I_LØ/Mtb_segmenter.cl"
mtb_segmenter = apoc.ObjectSegmenter(opencl_filename=mtb_cl_filename)

# Same folder list as batch notebook
main_directory_path = Path("X:\Lisa\siMtb screen I_LØ")

folders_with_nuc = [
    name for name in os.listdir(main_directory_path)
    if os.path.isdir(os.path.join(main_directory_path, name)) and "Nuc" not in name and "Results" not in name
]

# Pick one folder and one image for visualization
folder = folders_with_nuc[0]
directory_path = main_directory_path / folder
images = list_images(directory_path)
image = images[0]

print(f"Folder: {folder}")
print(f"Image: {image}")

# Markers for per-cell intensity (same as batch notebook)
markers = [("LC3B", 0), ("GAL3", 1), ("Chmp4B", 2), ("Mtb", 3)]
# Puncta markers (same as batch notebook)
puncta_markers = [("LC3B", 0), ("GAL3", 1), ("Chmp4B", 2)]

In [ ]:
slicing_factor_xy = None  # Use 2 or 4 for downsampling in xy (None for lossless)

# Brightfield correction (uses cached bf_correction.tiff if present)
bf_correction = brightfield_correction(directory_path, images, slicing_factor_xy)

# Read single image
img, filename = read_image(image, slicing_factor_xy)

plate_nr = filename.split("_")[0]
well_id = filename.split("Wells-")[1].split("__")[0]
print(f"Plate: {plate_nr}, Well: {well_id}")

In [ ]:
# Predict cytoplasm labels using CellposeSAM
# LC3B(img[0]) and GAL3(img[1]) are combined to create the "cytoplasm" channel, a marker agnostic input is the corrected brightfield image
# CellposeSAM is invariant to channel ordering 
cell_labels, flows, styles = model.eval(
    np.stack((img[[0, 1]].sum(axis=0), (img[4] - bf_correction)), axis=0
), niter=1000)

# Infection detection (same as batch notebook)
infection_stats = []
(
    mtb_labels,
    membrane_labels,
    cytoplasm_labels,
    infected_cell_labels,
    infected_membrane_labels,
    infected_cytoplasm_labels,
) = detect_infected_cells(
    img, mtb_segmenter, cell_labels, plate_nr, well_id, infection_stats
)

# Cell, membrane and cytoplasm labels restricted to Mtb-positive cells (for Napari overlay)
infected_cells_layer = np.where(
    np.isin(cell_labels, infected_cell_labels), cell_labels, 0
).astype(cell_labels.dtype)

infected_membranes_layer = np.where(
    np.isin(membrane_labels, infected_membrane_labels), membrane_labels, 0
).astype(membrane_labels.dtype)

infected_cytoplasms_layer = np.where(
    np.isin(cytoplasm_labels, infected_cytoplasm_labels), cytoplasm_labels, 0
).astype(cytoplasm_labels.dtype)

In [ ]:
# Per-image extraction pipeline (same as batch notebook; single image / folder)
mtb_props_df = extract_mtb_regionprops(mtb_labels, plate_nr, well_id, image)
mtb_props_df = map_bacterial_location(
    mtb_labels, cell_labels, membrane_labels, cytoplasm_labels, mtb_props_df
)

props_df = extract_cell_features(img, cell_labels, markers, plate_nr, well_id, image)
props_df = detect_infection_load(mtb_labels, cell_labels, props_df)
props_df, puncta_points = puncta_detection(
    img, puncta_markers, spotiflow_model, cell_labels, props_df, return_points=True
)

for name, pts in puncta_points.items():
    print(f"{name}: {len(pts)} spots")
print("Puncta points ready for Napari.")

props_df.rename(columns={"label": "ObjectNumber"}, inplace=True)
col_idx = props_df.columns.get_loc("ObjectNumber")
props_df.insert(col_idx + 1, "Mtb_infected_cell", props_df["ObjectNumber"].isin(infected_cell_labels))
props_df.insert(col_idx + 2, "Mtb_infected_membrane", props_df["ObjectNumber"].isin(infected_membrane_labels))
props_df.insert(col_idx + 3, "Mtb_infected_cytoplasm", props_df["ObjectNumber"].isin(infected_cytoplasm_labels))

df_infection = pd.DataFrame(infection_stats)

In [ ]:
import napari

viewer = napari.Viewer(ndisplay=2)

# Image layers: input channels used for segmentation and key signals
viewer.add_image(img, name="input_image")
viewer.add_image(img[4] - bf_correction, name="BF_corrected")
viewer.add_image(img[[0, 1]].sum(axis=0), name="cytoplasm_fluo_sum")
viewer.add_image(img[3], name="Mtb_ch")

# Labels
viewer.add_labels(cell_labels, name="cell_labels")
viewer.add_labels(membrane_labels, name="membrane_labels")
viewer.add_labels(cytoplasm_labels, name="cytoplasm_labels")
viewer.add_labels(mtb_labels, name="Mtb_labels")
viewer.add_labels(infected_cells_layer, name="infected_cells")
viewer.add_labels(infected_membranes_layer, name="infected_membranes")
viewer.add_labels(infected_cytoplasms_layer, name="infected_cytoplasms")

# Puncta points (distinct colors per marker)
colors = {"LC3B": "red", "GAL3": "yellow", "Chmp4B": "green"}
for name, pts in puncta_points.items():
    if len(pts) > 0:
        viewer.add_points(pts, face_color=colors.get(name, "white"), name=f"puncta_{name}")
    else:
        viewer.add_points(np.zeros((0, 2)), face_color=colors.get(name, "white"), name=f"puncta_{name}")

In [ ]:
from IPython.display import display

display(df_infection)
display(props_df)
display(mtb_props_df)